In [1]:
import pandas as pd

file_path = "Wildfire_Dataset_61Day_Holidays_n_doy_population.parquet"

df = pd.read_parquet(file_path, engine="pyarrow")

In [2]:
df.head()

,latitude,longitude,datetime,Wildfire,pr,rmax,rmin,sph,srad,tmmn,...,fm1000,erc,etr,pet,vpd,window_id,prox_to_holiday,population,doy_sin,doy_cos
0,25.26027,-80.80099,2016-04-10,0,0.0,91.2,44.7,0.01111,306.6,290.7,...,18.6,27.0,8.0,5.9,0.71,25.2602699992712_-80.80099_0,50,2.488104e-14,0.985948,-0.167052
1,25.26027,-80.80099,2016-04-11,0,0.0,100.0,51.0,0.01295,242.3,290.1,...,18.5,25.0,5.9,4.6,0.75,25.2602699992712_-80.80099_0,49,2.488104e-14,0.982927,-0.183998
2,25.26027,-80.80099,2016-04-12,0,0.0,100.0,45.2,0.01282,299.7,289.6,...,18.4,27.0,6.6,5.3,0.79,25.2602699992712_-80.80099_0,48,2.488104e-14,0.979614,-0.200891
3,25.26027,-80.80099,2016-04-13,0,0.0,100.0,47.0,0.01364,326.5,290.0,...,18.3,27.0,6.5,5.5,0.74,25.2602699992712_-80.80099_0,47,2.488104e-14,0.976011,-0.217723
4,25.26027,-80.80099,2016-04-14,0,1.0,100.0,49.7,0.01437,292.7,292.0,...,18.2,27.0,6.4,5.2,0.78,25.2602699992712_-80.80099_0,46,2.488104e-14,0.972118,-0.234491


In [5]:
import numpy as np

df["datetime"] = pd.to_datetime(df["datetime"])
window_df = df.sort_values(["window_id", "datetime"], kind="mergesort")

window_summary = window_df.groupby("window_id", sort=False).agg(
    label=("Wildfire", "last"),
    start_date=("datetime", "min"),
    end_date=("datetime", "max"),
)
window_summary["end_year"] = window_summary["end_date"].dt.year

def summarize_window_split(name: str, assignments: pd.Series, summary: pd.DataFrame) -> pd.DataFrame:
    stats = (
        summary.assign(split=assignments)
        .groupby("split")
        .agg(
            windows=("label", "size"),
            positive_rate=("label", "mean"),
            start_year=("end_year", "min"),
            end_year=("end_year", "max"),
        )
        .reindex(["train", "validate", "test"])
    )
    stats["split_name"] = name
    return stats.reset_index().rename(columns={"index": "split"})

def _stratified_partition(ids: np.ndarray, labels: np.ndarray, fractions: tuple[float, float, float], seed: int) -> dict:
    rng = np.random.default_rng(seed)
    split_names = ["train", "validate", "test"]
    allocation = {name: [] for name in split_names}

    for label in np.unique(labels):
        label_ids = ids[labels == label].copy()
        rng.shuffle(label_ids)
        raw_counts = np.array(fractions, dtype=float) * len(label_ids)
        counts = np.floor(raw_counts).astype(int)
        remainder = len(label_ids) - counts.sum()
        if remainder > 0:
            order = np.argsort(-(raw_counts - counts))
            for idx in order[:remainder]:
                counts[idx] += 1

        start = 0
        for split_name, count in zip(split_names, counts):
            allocation[split_name].extend(label_ids[start:start + count].tolist())
            start += count

    return allocation

def make_stratified_split(summary: pd.DataFrame, seed: int = 42) -> pd.Series:
    ids = summary.index.to_numpy()
    labels = summary["label"].to_numpy()
    allocation = _stratified_partition(ids, labels, fractions=(0.8, 0.1, 0.1), seed=seed)
    assignments = pd.Series(index=summary.index, dtype="object")
    for split_name, split_ids in allocation.items():
        assignments.loc[split_ids] = split_name
    return assignments

def make_year_sorted_split(summary: pd.DataFrame, train_frac: float = 0.8, val_frac: float = 0.1) -> tuple[pd.Series, dict]:
    year_counts = summary.groupby("end_year").size().sort_index()
    years = year_counts.index.to_list()
    total = int(year_counts.sum())
    cumulative = year_counts.cumsum()

    best = None
    for train_idx in range(len(years) - 2):
        train_count = int(cumulative.iloc[train_idx])
        for val_idx in range(train_idx + 1, len(years) - 1):
            val_count = int(cumulative.iloc[val_idx] - cumulative.iloc[train_idx])
            test_count = total - train_count - val_count
            if min(train_count, val_count, test_count) == 0:
                continue
            objective = abs(train_count / total - train_frac) + abs(val_count / total - val_frac)
            candidate = {
                "objective": objective,
                "train_end_year": years[train_idx],
                "validate_end_year": years[val_idx],
                "train_count": train_count,
                "validate_count": val_count,
                "test_count": test_count,
            }
            if best is None or candidate["objective"] < best["objective"]:
                best = candidate

    if best is None:
        raise ValueError("Unable to create non-empty chronological train/validate/test split.")

    assignments = pd.Series(index=summary.index, dtype="object")
    assignments.loc[summary["end_year"] <= best["train_end_year"]] = "train"
    assignments.loc[
        summary["end_year"].between(best["train_end_year"] + 1, best["validate_end_year"])
    ] = "validate"
    assignments.loc[summary["end_year"] > best["validate_end_year"]] = "test"
    return assignments, best

split_stratified = make_stratified_split(window_summary)
split_year_sorted, year_split_meta = make_year_sorted_split(window_summary)

window_splits = window_summary[["label", "start_date", "end_date", "end_year"]].copy()
window_splits["split_stratified"] = split_stratified
window_splits["split_year_sorted"] = split_year_sorted

assert window_splits["split_stratified"].notna().all()
assert window_splits["split_year_sorted"].notna().all()

df = df.merge(
    window_splits[["split_stratified", "split_year_sorted"]],
    left_on="window_id",
    right_index=True,
    how="left",
    validate="many_to_one",
)

split_summary = pd.concat(
    [
        summarize_window_split("stratified", split_stratified, window_summary),
        summarize_window_split("year_sorted", split_year_sorted, window_summary),
    ],
    ignore_index=True,
)

print("Year-sorted boundary selection:", year_split_meta)
split_summary


Year-sorted boundary selection: {'objective': 0.05429183697765794, 'train_end_year': 2020, 'validate_end_year': 2021, 'train_count': 97124, 'validate_count': 9431, 'test_count': 19440}


,split,windows,positive_rate,start_year,end_year,split_name
0,train,100797,0.259879,2014,2025,stratified
1,validate,12599,0.259862,2014,2025,stratified
2,test,12599,0.259862,2014,2025,stratified
3,train,97124,0.186504,2014,2020,year_sorted
4,validate,9431,0.359453,2021,2021,year_sorted
5,test,19440,0.578138,2022,2025,year_sorted


In [ ]:
window_df = df.sort_values(["window_id", "datetime"], kind="mergesort").copy()

feature_cols = [
    col for col in window_df.columns
    if col not in [
        "Wildfire",
        "window_id",
        "datetime",
        "split_stratified",
        "split_year_sorted",
    ]
]

X_by_window = {}
y_by_window = {}

for window_id, group in window_df.groupby("window_id", sort=False):
    group = group.sort_values("datetime", kind="mergesort")
    assert len(group) == 61, f"Expected 61 rows in {window_id}, found {len(group)}"
    X_by_window[window_id] = group.iloc[:-1][feature_cols].to_numpy(dtype=np.float32)
    y_by_window[window_id] = np.int64(group.iloc[-1]["Wildfire"])

def collect_split_arrays(assignments: pd.Series, split_name: str):
    split_window_ids = assignments[assignments == split_name].index.to_list()
    X_split = np.stack([X_by_window[window_id] for window_id in split_window_ids])
    y_split = np.array([y_by_window[window_id] for window_id in split_window_ids], dtype=np.int64)
    return X_split, y_split

shape_rows = []
for split_family, assignments in {
    "stratified": split_stratified,
    "year_sorted": split_year_sorted,
}.items():
    for split_name in ["train", "validate", "test"]:
        X_split, y_split = collect_split_arrays(assignments, split_name)
        shape_rows.append(
            {
                "split_family": split_family,
                "split": split_name,
                "X_shape": X_split.shape,
                "y_shape": y_split.shape,
                "positive_rate": float(y_split.mean()),
            }
        )

pd.DataFrame(shape_rows)


In [8]:
from pathlib import Path
import torch

output_dir = Path("training_splits")
output_dir.mkdir(exist_ok=True)

def build_split_payload(assignments: pd.Series) -> dict:
    payload = {
        "feature_cols": feature_cols,
        "sequence_length": 60,
    }

    for split_name in ["train", "validate", "test"]:
        split_window_ids = assignments[assignments == split_name].index.to_list()
        X_split, y_split = collect_split_arrays(assignments, split_name)
        payload[split_name] = {
            "X": torch.tensor(X_split, dtype=torch.float32),
            "y": torch.tensor(y_split, dtype=torch.long),
            "window_ids": split_window_ids,
        }

    return payload

exports = {
    "stratified": build_split_payload(split_stratified),
    "year_sorted": build_split_payload(split_year_sorted),
}

output_paths = {}
for split_family, payload in exports.items():
    output_path = output_dir / f"wildfire_61day_{split_family}_splits.pt"
    torch.save(payload, output_path)
    output_paths[split_family] = str(output_path)

output_paths


{'stratified': 'training_splits/wildfire_61day_stratified_splits.pt',
 'year_sorted': 'training_splits/wildfire_61day_year_sorted_splits.pt'}